# Traffic Sign Recognition using CNN (GTSRB)
**Deep Learning Semester Project**

**Name:** Muhammad Azghan  

Run the cells from top to bottom. Before starting: Runtime > Change runtime type > **T4 GPU**.


## Step 1: Download the dataset from Kaggle
Paste your Kaggle API token (starts with `KGAT_`) in the cell below. **Never commit a real token to GitHub.**

In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_KAGGLE_TOKEN_HERE'
!pip install -q -U kaggle
!kaggle datasets download -d meowmeowmeowmeowmeow/gtsrb-german-traffic-sign -p /content/gtsrb --unzip
!ls /content/gtsrb

## Step 2: Load images (resize to 32x32, normalize)

In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt
from PIL import Image

IMG = 32
root = '/content/gtsrb/Train'
X, y = [], []
for c in range(43):
    d = os.path.join(root, str(c))
    for f in os.listdir(d):
        try:
            img = Image.open(os.path.join(d, f)).convert('RGB').resize((IMG, IMG))
            X.append(np.array(img)); y.append(c)
        except Exception:
            pass
X = np.array(X, dtype='float32') / 255.0
y = np.array(y)
print('Images:', X.shape, 'Labels:', y.shape)

## Step 3: Explore the data (use these plots in your report)

In [ ]:
class_names = ["Speed limit (20km/h)","Speed limit (30km/h)","Speed limit (50km/h)","Speed limit (60km/h)","Speed limit (70km/h)","Speed limit (80km/h)","End of speed limit (80km/h)","Speed limit (100km/h)","Speed limit (120km/h)","No passing","No passing for vehicles over 3.5 tons","Right-of-way at next intersection","Priority road","Yield","Stop","No vehicles","Vehicles over 3.5 tons prohibited","No entry","General caution","Dangerous curve to the left","Dangerous curve to the right","Double curve","Bumpy road","Slippery road","Road narrows on the right","Road work","Traffic signals","Pedestrians","Children crossing","Bicycles crossing","Beware of ice/snow","Wild animals crossing","End of all speed and passing limits","Turn right ahead","Turn left ahead","Ahead only","Go straight or right","Go straight or left","Keep right","Keep left","Roundabout mandatory","End of no passing","End of no passing for vehicles over 3.5 tons"]
os.makedirs('outputs', exist_ok=True)
with open('outputs/class_names.json', 'w') as f:
    json.dump(class_names, f)

counts = np.bincount(y)
plt.figure(figsize=(14, 4))
plt.bar(range(43), counts)
plt.title('Images per class'); plt.xlabel('Class ID'); plt.ylabel('Count')
plt.savefig('outputs/class_distribution.png', dpi=150, bbox_inches='tight'); plt.show()

plt.figure(figsize=(12, 6))
idx = np.random.choice(len(X), 15, replace=False)
for i, k in enumerate(idx):
    plt.subplot(3, 5, i + 1)
    plt.imshow(X[k]); plt.title(class_names[y[k]][:18], fontsize=8); plt.axis('off')
plt.savefig('outputs/sample_images.png', dpi=150, bbox_inches='tight'); plt.show()

## Step 4: Train / validation / test split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)
print(X_train.shape, X_val.shape, X_test.shape)

## Step 5: Build the CNN
Augmentation is applied only during training (no horizontal flip, because flipping changes a sign's meaning).

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

aug = tf.keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
])

def block(m, filters):
    m.add(layers.Conv2D(filters, 3, padding='same', activation='relu'))
    m.add(layers.BatchNormalization())
    m.add(layers.Conv2D(filters, 3, padding='same', activation='relu'))
    m.add(layers.BatchNormalization())
    m.add(layers.MaxPooling2D())
    m.add(layers.Dropout(0.25))

model = models.Sequential([layers.Input(shape=(IMG, IMG, 3))])
block(model, 32); block(model, 64); block(model, 128)
model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(43, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## Step 6: Train (about 10 to 20 minutes on T4 GPU)

In [ ]:
train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(10000).batch(64)
            .map(lambda a, b: (aug(a, training=True), b), num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(64)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=25, callbacks=callbacks)

## Step 7: Evaluate and save plots for the report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc*100:.2f}%   Test loss: {test_loss:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'], label='train'); ax[0].plot(history.history['val_accuracy'], label='val')
ax[0].set_title('Accuracy'); ax[0].legend()
ax[1].plot(history.history['loss'], label='train'); ax[1].plot(history.history['val_loss'], label='val')
ax[1].set_title('Loss'); ax[1].legend()
plt.savefig('outputs/training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap='Blues')
plt.title('Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

report = classification_report(y_test, y_pred, target_names=class_names)
print(report)
with open('outputs/classification_report.txt', 'w') as f:
    f.write(f'Test accuracy: {test_acc*100:.2f}%\n\n' + report)

## Step 8: Save the model weights

In [ ]:
model.save('outputs/traffic_sign_model.keras')
model.save('outputs/traffic_sign_model.h5')
model.save_weights('outputs/traffic_sign.weights.h5')
!ls -lh outputs

## Step 9: Export the weights for the website
The browser demo (website/cnn.js) reads plain float32 weights, so no TensorFlow.js converter is needed.

In [ ]:
import os, json, numpy as np, tensorflow as tf

m = tf.keras.models.load_model('outputs/traffic_sign_model.keras')
os.makedirs('outputs/web_model', exist_ok=True)

meta, blob, offset = [], [], 0
for l in m.layers:
    entry = {'name': l.name, 'type': l.__class__.__name__, 'weights': []}
    cfg = l.get_config()
    if 'activation' in cfg:
        entry['activation'] = cfg['activation']
    if l.__class__.__name__ == 'BatchNormalization':
        entry['epsilon'] = float(l.epsilon)
    for a in l.get_weights():
        a = a.astype('float32')
        entry['weights'].append({'shape': list(a.shape), 'offset': offset})
        blob.append(a.ravel())
        offset += a.size
    meta.append(entry)

np.concatenate(blob).tofile('outputs/web_model/weights.bin')
with open('outputs/web_model/model_config.json', 'w') as f:
    json.dump({'input_shape': [32, 32, 3], 'layers': meta}, f)

!ls -lh outputs/web_model

## Step 1: Download the dataset from Kaggle
Paste your Kaggle API token (starts with `KGAT_`) in the cell below. **Never commit a real token to GitHub.**

In [ ]:
!zip -r outputs.zip outputs
from google.colab import files
files.download('outputs.zip')